# ATT GWAS

## Software and packages used

**Python version:** Python 3.8.6  

### Tools

Tools and versions used in this analysis:

| Tool        | Version             |
|:------------|:--------------------|
| Plink 1.9   | x86_64_20250615     |
| Plink 2.0   | linux_avx2_20250609 |
| Admix-kit   | 0.1.1               |

## Import packages and define variables

In [ ]:
import os
from concurrent.futures import ThreadPoolExecutor, as_completed
import subprocess

## Set directories and variables

#### Common paths

In [ ]:
# Hestia NGS Software
tools = "/path/to/tools"
# Main directory
main_dir = "/path/to/home"
# Data directory
data_dir = f"{main_dir}/DATA"
# Raw data directory
raw_dir = f"{data_dir}/RAW"
# Imputed data directory 
impt_dir = f"{data_dir}/IMPUTED/SOFTCALL_MAF005" # Prefiltered by maf
# Meta data (covariate, population, ancestry labels, etc.)
meta_dir = f"{data_dir}/META"
# Analysis directory
analysis = f"{data_dir}/ANALYSIS"

In [ ]:
# Local ancestry directory
la_dir = f"{data_dir}/IMPUTED/LA_inference_results"
# ATT GWAS
gwas_dir = f"{analysis}/ATT_GWAS"
os.makedirs(gwas_dir, exist_ok=True)

#### Paths to software and tools

In [ ]:
# Plink1.9 and Plink2.0 path
plink = f"{tools}/plink_linux_x86_64_20250615/plink"
plink2 = f"{tools}/plink2_linux_avx2_20250609/plink2"

#### Input, output, covariate files

In [ ]:
# Covariate file
covar = f"{meta_dir}/CATPD_unrel.cov"
pheno = f"{meta_dir}/CATPD_unrel.pheno"
# Keep unrelated
keep_file  = f"{meta_dir}/CATPD_unrel.keep"
# Chromosomes as list
chromosomes = list(range(1,23))

In [ ]:
def runGWAS(args):
    inputPfile, outputPfile, inputVCF, unrelated_list, threads, chnum, msp_in, msp_out = args

    # Get unrelated individuals
    get_unrelated = [
        "plink2",
        "--pfile", inputPfile,
        "--keep", unrelated_list,
        "--maf", "0.05",
        "--mac", str(20),
        "--max-alleles", "2",
        "--rm-dup", "exclude-all",
        "--snps-only",
        "--silent",
        "--threads", str(threads),
        "--make-pgen",
        "--out", inputPfile
    ]
    subprocess.run(get_unrelated, check=True)

    newmsp = [
        "python3", f"{main_dir}/newMSP.py",
        "-p", f"{inputPfile}.psam",
        "-m", f"{msp_in}",
        "-o", f"{msp_out}"
    ]
    subprocess.run(newmsp, check=True)
    
    # LANC convert
    convert = [
        "conda", "run", "-n", "admixkit_386",
        "admix", "lanc-convert", 
        "--pfile", f"{inputPfile}",
        "--rfmix", f"{msp_out}",
        "--out", f"{inputPfile}.lanc"
    ]
    subprocess.run(convert, check=True)

    # Run ATT
    for method in ["ATT"]:
        run_gwas = [
            "conda", "run", "-n", "admixkit_386",
            "admix", "assoc",
            "--pfile", inputPfile,
            "--pheno", covar,
            "--family", "binary",
            "--method", method,
            "--quantile-normalize", "True",
            "--out",  f"{analysis}/{method.upper()}_GWAS/chrom{chnum}_{method.upper()}"
        ]
        subprocess.run(run_gwas, check=True) 

In [ ]:
%%time
# Run thread pool executor
tasks = []

for chnum in chromosomes:
    inputPfile  = f"{gwas_dir}/chr{chnum}.unrelated"
    outputPfile = f"{gwas_dir}/chr{chnum}.unrelated_ATT"
    inputVCF    = f"{imputed_dir}/chr{chnum}.filtered.vcf"
    unrelated_list = keep_file
    threads = str(2)
    chnum = chnum
    msp_in = f"{la_dir}/model_chm{chnum}/query_results.msp"
    msp_out = f"{la_dir}/model_chm{chnum}/MSP_{chnum}_NEW.msp"
    tasks.append((inputPfile, outputPfile, inputVCF, unrelated_list, threads, chnum, msp_in, msp_out))
    
with ThreadPoolExecutor(max_workers=4) as ex:
    futures = [ex.submit(runGWAS, t) for t in tasks]
    for f in as_completed(futures):
        try:
            f.result()
        except Exception as e:
            print(f"Task failed: {e}")
            traceback.print_exc()